# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### My response
**Unit of analysis:** One row represents one content page for one client for one monthly observation window.

**Time window:** March 2026 will be used as the development month.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!pip -q install duckdb huggingface_hub
import duckdb
from google.colab import userdata
from huggingface_hub import login

# Read HF token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Log in to Hugging Face
login(token=HF_TOKEN)

# Create DuckDB connection
con = duckdb.connect()

print("Hugging Face login successful.")
print("DuckDB connection created.")

Hugging Face login successful.
DuckDB connection created.


In [2]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=HF_TOKEN)

files = fs.glob("datasets/FlyRank/internship-warehouse/**")

# Show only March 2026 files
march_files = [f for f in files if "2026-03" in f]

march_files[:20]

['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']

In [3]:
import pyarrow.fs as pafs
import pyarrow.dataset as ds

hf_fs = pafs.PyFileSystem(
    pafs.FSSpecHandler(fs)
)

march_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

march_dataset = ds.dataset(
    march_path,
    filesystem=hf_fs,
    format="parquet"
)

con.register("march_data", march_dataset)

print("March 2026 data connected successfully.")

March 2026 data connected successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### My response

**Features**
- `gsc_avg_position`: search position available for the page at the decision moment.
- `ga4_pageviews`: pageview volume available at the decision moment.
- `ga4_sessions`: session volume available at the decision moment.
- `ga4_engaged_sessions`: engaged-session volume available at the decision moment.
- `scroll_events`: observed scroll activity available at the decision moment.

**Label / proxy**
- `ctr_gap`: expected CTR based on the page's position/context minus its observed CTR.
- A larger positive gap indicates that the page may be under-capturing clicks relative to its expected level.

**Context**
- `client_hash_id`: identifies the client and is used for grouping/splitting, not as a model feature.
- `content_hash_id`: identifies the content page and is used for grouping/joining, not as a model feature.
- `report_date`: identifies the date of the observation.
- `gsc_data_available`: indicates whether GSC data is available for the observation.
- `ga4_data_available`: indicates whether GA4 data is available for the observation.

**Excluded**
- `gsc_clicks`: excluded because clicks are used to calculate observed CTR, which is part of the CTR-gap proxy.
- `gsc_impressions`: excluded from the feature set because impressions are also used to calculate observed CTR and could make the ranking too closely tied to the proxy.
- Future-window metrics: excluded because they would not be available at the decision moment.
- Client and content identifiers: excluded as model features because they are identifiers rather than meaningful performance signals.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the columns available in the March 2026 daily performance data
march_dataset.schema

report_date: date32[day]
client_hash_id: string
content_hash_id: string
client_has_gsc: bool
client_has_ga4: bool
gsc_data_available: bool
ga4_data_available: bool
gsc_impressions: int64
gsc_clicks: int64
gsc_sum_position: int64
gsc_avg_position: double
ga4_pageviews: int64
ga4_sessions: int64
ga4_users: int64
ga4_engaged_sessions: int64
ga4_total_engagement_sec: int64
sessions_organic: int64
sessions_direct: int64
sessions_referral: int64
sessions_social: int64
sessions_paid: int64
sessions_ai: int64
ai_chatgpt: int64
ai_perplexity: int64
ai_gemini: int64
ai_copilot: int64
ai_claude: int64
ai_meta: int64
ai_other: int64
scroll_events: int64

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification summary

- **Grain:** No duplicate `report_date + client_hash_id + content_hash_id` combinations were found in March 2026, supporting the expected daily client-content grain.
- **Window and counts:** The March 2026 partition contains 984,137 rows, covering 2026-03-01 to 2026-03-31, across 55 clients and 331,437 content pages.
- **Data availability:** GSC and GA4 availability were checked using their explicit availability flags with `IS TRUE`. GA4 availability is not assumed from zero-valued metrics.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
# Query 1: Check for duplicate rows at the expected daily grain

con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM march_data
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [7]:
# Query 2: Check row count and date range

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_data
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬───────────────┬────────────┬────────────┐
│ total_rows │ clients │ content_pages │ first_date │ last_date  │
│   int64    │  int64  │     int64     │    date    │    date    │
├────────────┼─────────┼───────────────┼────────────┼────────────┤
│    9841378 │      55 │        331437 │ 2026-03-01 │ 2026-03-31 │
└────────────┴─────────┴───────────────┴────────────┴────────────┘

In [8]:
# Query 3: Check GSC and GA4 data availability

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM march_data
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### My response

- **Unbalanced history:** Different clients have different amounts of historical data, so March 2026 does not represent the same history depth for every client.
- **GSC-only early rows:** Some rows occur before a client's GA4 tracking started. When `ga4_data_available = FALSE`, zero-valued GA4 fields cannot be interpreted as zero engagement.
- **Window overlap:** Features and labels must use non-overlapping time windows. A feature that includes information from the label period would leak future information into the ranking.
- **Measurement limitation:** The data can identify observed CTR and engagement patterns, but it cannot by itself prove that changing a page will cause higher CTR or engagement.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.